In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import gc
from tqdm import tqdm
import pickle
import psutil

In [ ]:
PROJECT_ROOT = Path("D:/Project/Radiology_RAG")
UMLS_META = PROJECT_ROOT / "data/raw_data/umls/umls_2026AA_full/2026AA/META"
OUTPUT_DIR = PROJECT_ROOT / "data/processed_data/umls/try04"
MRCONSO_FILTERED = Path("D:/Project/Radiology_RAG/data/processed_data/filtered_data/MRCONSO.RRF")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
CHEST_KEYWORDS = [
    # # Anatomy - specific chest structures only
    # r'\blung', r'\blungs', r'\bchest', r'\bthorax', r'\bthoracic',
    # r'\bheart\b', r'\bcardiac', r'\bmediastin', r'\bpleura', r'\bpleural',
    # r'\brib\b', r'\bribs\b', r'\bintercost', r'\bdiaphragm',
    # r'\bbronch', r'\btrachea', r'\bpulmonary', r'\btracheal',
    
    # # Chest-specific pathology (avoid generic terms!)
    # r'\bpneumonia', r'\bpneumothorax', r'\bbronchitis',
    # r'\bemphysema', r'\bpulmonary edema', r'\batelectasis',
    # r'\bpleural effusion', r'\binfiltrate', r'\bconsolidation',
    # r'\bcardiomegaly', r'\bpulmonary nodule', r'\bcavitary',
    # r'\bfibrosis', r'\btubercul', r'\bsarcoidosis',
    # r'\basthma', r'\bcopd', r'\bcystic fibrosis', r'\baspiration pneumonia',
    
    # # Chest-specific findings (NOT generic "finding")
    # r'\bopacity\b', r'\bopacities', r'\binfiltrate', r'\bconsolidation',
    # r'\batelectatic', r'\bhyperinflation', r'\bflattening', r'\bdeviation'

    # Chest/Thorax anatomy
    r'\blung\b', r'\blungs\b', r'\bpulmonary\b',
    r'\bthorax\b', r'\bthoracic\b', r'\bchest\b',
    r'\bmediastin', r'\bpleura', r'\bpleural\b',
    r'\bdiaphragm\b', r'\bribs?\b', r'\btrachea\b',
    
    # Pathology - chest-specific
    r'\bpneumonia\b', r'\bpneumothorax\b',
    r'\bconsolidation\b', r'\binfiltrat',
    r'\batelectasis\b', r'\bemphysema\b',
    r'\bfibrosis\b', r'\bcavit',
    r'\bpleural effusion\b', r'\bpulmonary edema\b',
    r'\btuberculosis\b', r'\btubercul',
    
    # Imaging findings
    r'\bopacit', r'\bhyperinflation\b',
    r'\bpulmonary nodule\b', r'\bpulmonary mass\b',
    r'\bsarcoidosis\b', r'\bcystic fibrosis\b',
    
    # Cardiac-specific
    r'\bcardiomegaly\b', r'\bcardiac silhouette\b',
    r'\bhilum\b', r'\bhila\b', r'\bhilar\b',

]

CHUNK_SIZE = 5000
CHEST_RELEVANT_STYPES = {
    'T003',  # Body Part, Organ, or Organ Component
    'T004',  # Body Substance
    'T005',  # Disease or Syndrome
    'T007',  # Finding
}

pattern = '|'.join(CHEST_KEYWORDS)


# UMLS

## Extract Chest Radiology CUIs

In [ ]:
def extract_chest_cuis():
    """
    Extract all CUIs related to chest radiology from MRCONSO.RRF
    Save as CSV with metadata.
    
    Returns:
        DataFrame: CUIs with extraction details
        set: CUI set for next steps
    """
    
    mrconso_file = MRCONSO_FILTERED
    
    if not mrconso_file.exists():
        raise FileNotFoundError(f"MRCONSO.RRF not found at {mrconso_file}")
    
    print(f"Reading MRCONSO.RRF ({mrconso_file.stat().st_size / 1e9:.1f} GB)...")
    print(f"Chunk size: {CHUNK_SIZE:,} rows")
    
    chest_cuis_list = []  # Will be list of dicts for DataFrame
    chest_cuis_set = set()  # For filtering in next steps
    total_rows_processed = 0

    
    # Read in chunks to avoid RAM overload
    for chunk_num, chunk in enumerate(
        pd.read_csv(
            mrconso_file,
            sep='|',
            header=None,
            usecols=[0, 14],  # CUI (col 0) and STR (col 14)
            names=['CUI', 'STR'],
            chunksize=CHUNK_SIZE,
            dtype={'CUI': str, 'STR': str},
            engine='c'
        ),
        1
    ):
        # Convert term strings to lowercase for matching
        chunk['STR_lower'] = chunk['STR'].str.lower()
        
        # Find rows matching any chest keyword
        mask = chunk['STR_lower'].str.contains(
            '|'.join(CHEST_KEYWORDS),
            na=False,
            regex=True
        )
        
        # Extract CUIs and their matching terms
        matching = chunk[mask].copy()
        
        for _, row in matching.iterrows():
            cui = row['CUI']
            if cui not in chest_cuis_set:
                chest_cuis_set.add(cui)
        
        total_rows_processed += len(chunk)
        
        if chunk_num % 5 == 0:
            print(f"  Processed {total_rows_processed:,} rows | "
                  f"Found {len(chest_cuis_set):,} keyword-matched CUIs so far...")
        
        del chunk, mask, matching
        gc.collect()
    
    print(f"\n✓ Stage 1 (Keyword): {len(chest_cuis_set):,} CUIs")
    
    # STAGE 2: Will be validated by semantic type in next step
    # Save preliminary list
    cuis_df = pd.DataFrame({'CUI': sorted(list(chest_cuis_set))})
    cuis_csv = OUTPUT_DIR / "01_chest_radiology_cuis_preliminary.csv"
    cuis_df.to_csv(cuis_csv, index=False)
    print(f"✓ Saved preliminary CUIs (before semantic type filtering): {cuis_csv}")
    print(f"\nNote: These will be further filtered by semantic type (T003, T005, T007)")
    print(f"       when loaded in STEP 3 (Semantic Types)")
    
    return cuis_df, chest_cuis_set



In [ ]:
def extract_and_filter_cuis():
    """
    Two-stage filtering:
    Stage 1: Keyword match (with word boundaries)
    Stage 2: Semantic type validation (T003/T004/T005/T007)
    """
    print("\n" + "="*70)
    print("STEP 1: EXTRACT CHEST CUIs (WITH SEMANTIC TYPE FILTERING)")
    print("="*70)
    
    # First pass: Load ALL semantic types into memory
    print("\n1a. Pre-loading all semantic types into memory...")
    mrsty_file = UMLS_META / "MRSTY.RRF"
    
    all_stypes = {}
    for chunk in pd.read_csv(
        mrsty_file,
        sep='|',
        header=None,
        usecols=[0, 2],  # ← FIXED: Column 2 is STY (code), not column 3
        names=['CUI', 'STY'],
        chunksize=CHUNK_SIZE,
        dtype={'CUI': str, 'STY': str},
        engine='c'
    ):
        for _, row in chunk.iterrows():
            cui = row['CUI'].strip()
            sty = row['STY'].strip()
            if cui not in all_stypes:
                all_stypes[cui] = []
            all_stypes[cui].append(sty)
    
    print(f"   Loaded semantic types for {len(all_stypes):,} CUIs")
    
    # Second pass: Keyword filter + semantic type check
    print("\n1b. Keyword filtering with semantic type validation...")
    mrconso_file = MRCONSO_FILTERED
    
    chest_cuis_list = []
    chest_cuis_set = set()
    total_rows = 0
    keyword_matches = 0
    semantic_validated = 0
    
    for chunk_num, chunk in enumerate(
        pd.read_csv(
            mrconso_file,
            sep='|',
            header=None,
            usecols=[0, 14],
            names=['CUI', 'STR'],
            chunksize=CHUNK_SIZE,
            dtype={'CUI': str, 'STR': str},
            engine='c'
        ),
        1
    ):
        chunk['STR_lower'] = chunk['STR'].str.lower()
        
        # Stage 1: Keyword match (with word boundaries and regex)
        keyword_mask = chunk['STR_lower'].str.contains(
            pattern, 
            na=False, 
            regex=True,  # ← CRITICAL: enables \b word boundaries
            case=False
        )
        keyword_matches += keyword_mask.sum()
        
        # Stage 2: Check if CUI has chest-relevant semantic type
        matching = chunk[keyword_mask].copy()
        
        for _, row in matching.iterrows():
            cui = row['CUI']
            term = row['STR']
            
            # Only add if CUI has T003/T004/T005/T007
            if cui in all_stypes:
                stypes = all_stypes[cui]
                has_chest_type = any(st in CHEST_RELEVANT_STYPES for st in stypes)
                
                if has_chest_type and cui not in chest_cuis_set:
                    chest_cuis_list.append({
                        'CUI': cui,
                        'first_matching_term': term,
                        'semantic_types': ', '.join(stypes)
                    })
                    chest_cuis_set.add(cui)
                    semantic_validated += 1
        
        total_rows += len(chunk)
        
        if chunk_num % 5 == 0:
            print(f"  Processed {total_rows:,} rows | "
                  f"Keyword matches: {keyword_matches:,} | "
                  f"Validated: {semantic_validated:,}")
        
        del chunk, keyword_mask, matching
        gc.collect()
    
    # Convert to DataFrame
    cuis_df = pd.DataFrame(chest_cuis_list)
    cuis_df = cuis_df.sort_values('CUI').reset_index(drop=True)
    
    print(f"\n" + "="*70)
    print("FILTERING RESULTS:")
    print("="*70)
    print(f"Total rows scanned:           {total_rows:,}")
    print(f"Keyword matches:              {keyword_matches:,}")
    print(f"After semantic type filter:   {semantic_validated:,}")
    print(f"Reduction from keywords:      {100 * (1 - semantic_validated/keyword_matches):.1f}%")
    
    # Save
    cuis_csv = OUTPUT_DIR / "01_chest_radiology_cuis_filtered.csv"
    cuis_df.to_csv(cuis_csv, index=False)
    print(f"\n✓ Saved: {cuis_csv}")
    
    # Show sample
    print(f"\nSample CUIs:")
    print(cuis_df.head(10).to_string(index=False))
    
    return cuis_df, chest_cuis_set

## Load Definitions (MRDEF)

In [ ]:
def load_definitions(chest_cuis_set):
    """
    Load definitions from MRDEF.RRF for chest CUIs.
    Save as CSV with one row per CUI + source combination.
    
    Args:
        chest_cuis_set (set): CUIs to load definitions for
    
    Returns:
        DataFrame: Definitions in tabular form
    """

    
    mrdef_file = UMLS_META / "MRDEF.RRF"
    
    if not mrdef_file.exists():
        raise FileNotFoundError(f"MRDEF.RRF not found at {mrdef_file}")
    
    print(f"Reading MRDEF.RRF ({mrdef_file.stat().st_size / 1e9:.1f} GB)...")
    
    definitions_list = []
    total_rows = 0
    chv_count = 0

    
    for chunk_num, chunk in enumerate(
        pd.read_csv(
            mrdef_file,
            sep='|',
            header=None,
            usecols=[0, 4, 5],  # CUI, SAB (source), DEF (definition)
            names=['CUI', 'SAB', 'DEF'],
            chunksize=CHUNK_SIZE,
            dtype={'CUI': str, 'SAB': str, 'DEF': str},
            engine='c'
        ),
        1
    ):
        # Filter to chest CUIs
        filtered = chunk[chunk['CUI'].isin(chest_cuis_set)]
        
        if len(filtered) > 0:
            for _, row in filtered.iterrows():
                definitions_list.append({
                    'CUI': row['CUI'],
                    'source': row['SAB'],
                    'definition': row['DEF']
                })
                if row['SAB'] == 'CHV':
                    chv_count +=1
        
        total_rows += len(chunk)
        
        if chunk_num % 5 == 0:
            print(f"  Processed {total_rows:,} rows | "
                  f"Found {len(definitions_list):,} definitions")
        
        del chunk, filtered
        gc.collect()
    
    # Convert to DataFrame
    defs_df = pd.DataFrame(definitions_list)
    
   # CHV ANALYSIS
    print(f"\n" + "="*70)
    print(" DEFINITION COVERAGE ANALYSIS")
    print("="*70)
    
    source_counts = defs_df['source'].value_counts()
    print(f"\nDefinitions by source (top 10):")
    for source, count in source_counts.head(10).items():
        pct = 100 * count / len(defs_df)
        print(f"  {source:15s}: {count:6,} ({pct:5.1f}%)")
    
    chv_cuis = defs_df[defs_df['source'] == 'CHV']['CUI'].nunique()
    print(f"\n🚨 CHV Coverage:")
    print(f"  Total CHV definitions: {chv_count:,}")
    print(f"  CUIs with CHV defs:    {chv_cuis:,} / {len(chest_cuis_set):,} ({100*chv_cuis/len(chest_cuis_set):.1f}%)")
    
    if chv_cuis < len(chest_cuis_set) * 0.3:
        print(f"\n  WARNING: CHV coverage is LOW ({100*chv_cuis/len(chest_cuis_set):.1f}%)")
        print(f"   Recommendation: Use MSH definitions as fallback")
    
    # Save
    defs_csv = OUTPUT_DIR / "02_definitions.csv"
    defs_df.to_csv(defs_csv, index=False)
    print(f"\n✓ Saved: {defs_csv}")
    
    return defs_df

In [ ]:
def load_definitions(chest_cuis_set):
    """
    Load definitions. Report on CHV coverage.
    """
    print("\n" + "="*70)
    print("STEP 2: LOADING DEFINITIONS (MRDEF)")
    print("="*70)
    
    mrdef_file = UMLS_META / "MRDEF.RRF"
    
    definitions_list = []
    total_rows = 0
    chv_count = 0
    
    for chunk_num, chunk in enumerate(
        pd.read_csv(
            mrdef_file,
            sep='|',
            header=None,
            usecols=[0, 4, 5],
            names=['CUI', 'SAB', 'DEF'],
            chunksize=CHUNK_SIZE,
            dtype={'CUI': str, 'SAB': str, 'DEF': str},
            engine='c'
        ),
        1
    ):
        filtered = chunk[chunk['CUI'].isin(chest_cuis_set)]
        
        if len(filtered) > 0:
            for _, row in filtered.iterrows():
                definitions_list.append({
                    'CUI': row['CUI'],
                    'source': row['SAB'],
                    'definition': row['DEF']
                })
                if row['SAB'] == 'CHV':
                    chv_count += 1
        
        total_rows += len(chunk)
        
        if chunk_num % 5 == 0:
            print(f"  Processed {total_rows:,} rows | Found {len(definitions_list):,} defs so far...")
        
        del chunk, filtered
        gc.collect()
    
    defs_df = pd.DataFrame(definitions_list)
    
    # CHV ANALYSIS
    print(f"\n" + "="*70)
    print("⚠️  DEFINITION COVERAGE ANALYSIS")
    print("="*70)
    
    source_counts = defs_df['source'].value_counts()
    print(f"\nDefinitions by source (top 10):")
    for source, count in source_counts.head(10).items():
        pct = 100 * count / len(defs_df)
        print(f"  {source:15s}: {count:6,} ({pct:5.1f}%)")
    
    chv_cuis = defs_df[defs_df['source'] == 'CHV']['CUI'].nunique()
    print(f"\n🚨 CHV Coverage:")
    print(f"  Total CHV definitions: {chv_count:,}")
    print(f"  CUIs with CHV defs:    {chv_cuis:,} / {len(chest_cuis_set):,} ({100*chv_cuis/len(chest_cuis_set):.1f}%)")
    
    if chv_cuis < len(chest_cuis_set) * 0.3:
        print(f"\n⚠️  WARNING: CHV coverage is LOW ({100*chv_cuis/len(chest_cuis_set):.1f}%)")
        print(f"   Will use MSH definitions as fallback")
    
    # Save
    defs_csv = OUTPUT_DIR / "02_definitions.csv"
    defs_df.to_csv(defs_csv, index=False)
    print(f"\n✓ Saved: {defs_csv}")
    
    return defs_df


## Load Semantic Types (MRSTY)

In [ ]:
def load_semantic_types(chest_cuis_set):
    """
    Load semantic types from MRSTY.RRF for chest CUIs
    Semantic types categorize concepts (e.g., Disease, Finding, Anatomy)
    Save as CSV with one row per CUI + semantic type combination.

    Args:
        chest_cuis_set : CUIs to load semantic types for
    
    Returns:
        DataFrame: Semantic types in tabular form
    """
    
    mrsty_file = UMLS_META / "MRSTY.RRF"
    
    if not mrsty_file.exists():
        raise FileNotFoundError(f"MRSTY.RRF not found at {mrsty_file}")
    
    print(f"Reading MRSTY.RRF ({mrsty_file.stat().st_size / 1e9:.1f} GB)...")
    
    stypes_list = []
    total_rows = 0
    
    for chunk_num, chunk in enumerate(
        pd.read_csv(
            mrsty_file,
            sep='|',
            header=None,
            usecols=[0, 3],  # CUI, STY (semantic type)
            names=['CUI', 'STY'],
            chunksize=CHUNK_SIZE,
            dtype={'CUI': str, 'STY': str},
            engine='c'
        ),
        1
    ):
        filtered = chunk[chunk['CUI'].isin(chest_cuis_set)]
        
        if len(filtered) > 0:
            for _, row in filtered.iterrows():
                stypes_list.append({
                    'CUI': row['CUI'],
                    'semantic_type': row['STY']
                })
        
        total_rows += len(chunk)
        
        if chunk_num % 5 == 0:
            print(f"  Processed {total_rows:,} rows | "
                  f"Found {len(stypes_list):,} semantic types so far...")
        
        del chunk, filtered
        gc.collect()
    
    # Convert to DataFrame
    stypes_df = pd.DataFrame(stypes_list)
    
    print(f"\n✓ Total semantic type entries: {len(stypes_df):,}")
    print(f"✓ Total rows processed: {total_rows:,}")
    
    # Show type distribution
    print(f"\n✓ Semantic type distribution:")
    type_counts = stypes_df['semantic_type'].value_counts()
    for stype, count in type_counts.items():
        print(f"    {stype}: {count:,}")
    
    # Save to CSV
    stypes_csv = OUTPUT_DIR / "03_semantic_types.csv"
    stypes_df.to_csv(stypes_csv, index=False)
    print(f"\n✓ Saved: {stypes_csv}")
    
    return stypes_df

In [ ]:
def load_semantic_types(chest_cuis_set):
    """
    Load semantic types and FILTER to chest-relevant types.
    """
    
    mrsty_file = UMLS_META / "MRSTY.RRF"

    
    print(f"Reading MRSTY.RRF ({mrsty_file.stat().st_size / 1e9:.1f} GB)...")
    print(f"Filtering to semantic types: {CHEST_RELEVANT_STYPES}")
    
    stypes_list = []
    total_rows = 0
    valid_cuis = set()  # Track which CUIs have chest-relevant types
    
    for chunk_num, chunk in enumerate(
        pd.read_csv(
            mrsty_file,
            sep='|',
            header=None,
            usecols=[0, 3],
            names=['CUI', 'STY'],
            chunksize=CHUNK_SIZE,
            dtype={'CUI': str, 'STY': str},
            engine='c'
        ),
        1
    ):
        # Only keep rows that:
        # 1. CUI is in our keyword-matched set
        # 2. Semantic type is chest-relevant
        filtered = chunk[
            (chunk['CUI'].isin(chest_cuis_set)) &
            (chunk['STY'].isin(CHEST_RELEVANT_STYPES))
        ]
        
        if len(filtered) > 0:
            for _, row in filtered.iterrows():
                stypes_list.append({
                    'CUI': row['CUI'],
                    'semantic_type': row['STY']
                })
                valid_cuis.add(row['CUI'])
        
        total_rows += len(chunk)
        
        if chunk_num % 5 == 0:
            print(f"  Processed {total_rows:,} rows | "
                  f"Found {len(valid_cuis):,} CUIs with chest-relevant types...")
        
        del chunk, filtered
        gc.collect()
    
    stypes_df = pd.DataFrame(stypes_list)
    
    # IMPORTANT: Update chest_cuis_set to only include CUIs with valid semantic types
    # This removes false positives like "Maghrib", "abnormal", etc.
    original_count = len(chest_cuis_set)
    chest_cuis_set_filtered = valid_cuis
    removed = original_count - len(chest_cuis_set_filtered)
    
    print(f"\n✓ Semantic type filtering results:")
    print(f"  Before: {original_count:,} CUIs (from keyword match)")
    print(f"  After:  {len(chest_cuis_set_filtered):,} CUIs (with T003/T004/T005/T007)")
    print(f"  Removed: {removed:,} false positives ({100*removed/original_count:.1f}%)")
    
    stypes_csv = OUTPUT_DIR / "03_semantic_types.csv"
    stypes_df.to_csv(stypes_csv, index=False)
    print(f"✓ Saved: {stypes_csv}")
    
    return stypes_df, chest_cuis_set_filtered

## Load Preferred Terms (MRCONSO)

In [ ]:
def load_preferred_terms(chest_cuis_set):
    """
    Load preferred terms and all synonyms from MRCONSO.RRF.
    Save as CSV with one row per CUI + term combination.
    
    Args:
        chest_cuis_set (set): CUIs to load terms for
    
    Returns:
        DataFrame: Terms in tabular form
    """
    
    mrconso_file = MRCONSO_FILTERED
    
    print(f"Reading MRCONSO.RRF for term extraction...")
    
    terms_list = []
    total_rows = 0
    
    for chunk_num, chunk in enumerate(
        pd.read_csv(
            mrconso_file,
            sep='|',
            header=None,
            usecols=[0, 6, 14],  # CUI, ISPREF (is preferred?), STR (term)
            names=['CUI', 'ISPREF', 'STR'],
            chunksize=CHUNK_SIZE,
            dtype={'CUI': str, 'ISPREF': str, 'STR': str},
            engine='c'
        ),
        1
    ):
        # Filter to chest CUIs
        filtered = chunk[chunk['CUI'].isin(chest_cuis_set)]
        
        if len(filtered) > 0:
            for _, row in filtered.iterrows():
                terms_list.append({
                    'CUI': row['CUI'],
                    'term': row['STR'],
                    'is_preferred': row['ISPREF'] == 'Y'
                })
        
        total_rows += len(chunk)
        
        if chunk_num % 5 == 0:
            print(f"  Processed {total_rows:,} rows | "
                  f"Found {len(terms_list):,} terms so far...")
        
        del chunk, filtered
        gc.collect()
    
    # Convert to DataFrame
    terms_df = pd.DataFrame(terms_list)
    
    print(f"\n✓ Total term entries: {len(terms_df):,}")
    print(f"✓ Total rows processed: {total_rows:,}")
    
    # Count preferred vs synonyms
    preferred_count = terms_df['is_preferred'].sum()
    synonym_count = len(terms_df) - preferred_count
    
    print(f"✓ Preferred terms: {preferred_count:,}")
    print(f"✓ Synonym terms: {synonym_count:,}")
    
    # Save to CSV
    terms_csv = OUTPUT_DIR / "04_terms.csv"
    terms_df.to_csv(terms_csv, index=False)
    print(f"\n✓ Saved: {terms_csv}")
    
    # Show sample
    print(f"\nSample terms:")
    sample = terms_df[terms_df['is_preferred']].head(10)
    print(sample.to_string(index=False))
    
    return terms_df


In [ ]:
def load_preferred_terms(chest_cuis_set):
    """
    Load all terms for chest CUIs.
    """
    print("\n" + "="*70)
    print("STEP 3: LOADING TERMS AND SYNONYMS")
    print("="*70)
    
    mrconso_file = MRCONSO_FILTERED
    
    terms_list = []
    total_rows = 0
    
    for chunk_num, chunk in enumerate(
        pd.read_csv(
            mrconso_file,
            sep='|',
            header=None,
            usecols=[0, 6, 14],
            names=['CUI', 'ISPREF', 'STR'],
            chunksize=CHUNK_SIZE,
            dtype={'CUI': str, 'ISPREF': str, 'STR': str},
            engine='c'
        ),
        1
    ):
        filtered = chunk[chunk['CUI'].isin(chest_cuis_set)]
        
        if len(filtered) > 0:
            for _, row in filtered.iterrows():
                terms_list.append({
                    'CUI': row['CUI'],
                    'term': row['STR'],
                    'is_preferred': row['ISPREF'] == 'Y'
                })
        
        total_rows += len(chunk)
        
        if chunk_num % 5 == 0:
            print(f"  Processed {total_rows:,} rows | Found {len(terms_list):,} terms so far...")
        
        del chunk, filtered
        gc.collect()
    
    terms_df = pd.DataFrame(terms_list)
    
    preferred_count = terms_df['is_preferred'].sum()
    
    print(f"\nTerms found:")
    print(f"  Total term entries:  {len(terms_df):,}")
    print(f"  Preferred terms:     {preferred_count:,}")
    print(f"  Synonym terms:       {len(terms_df) - preferred_count:,}")
    print(f"  Avg synonyms/CUI:    {len(terms_df) / len(chest_cuis_set):.1f}")
    
    terms_csv = OUTPUT_DIR / "03_terms.csv"
    terms_df.to_csv(terms_csv, index=False)
    print(f"\n✓ Saved: {terms_csv}")
    
    return terms_df


## Summary & Execution

In [ ]:
def print_summary_report(cuis_df, defs_df, stypes_df, terms_df):
    """
    Print summary statistics across all extracted data.
    """
    print("\n" + "="*70)
    print("EXTRACTION SUMMARY REPORT")
    print("="*70)
    
    # CUI statistics
    print(f"\n1. CUIS:")
    print(f"   Total CUIs: {len(cuis_df):,}")
    
    # Definition statistics
    print(f"\n2. DEFINITIONS:")
    print(f"   Total definition entries: {len(defs_df):,}")
    cuis_with_defs = defs_df['CUI'].nunique()
    print(f"   Unique CUIs with definitions: {cuis_with_defs:,}")
    print(f"   Percent covered: {100*cuis_with_defs/len(cuis_df):.1f}%")
    print(f"   Definition sources: {defs_df['source'].nunique()}")
    
    # Semantic type statistics
    print(f"\n3. SEMANTIC TYPES:")
    print(f"   Total semantic type entries: {len(stypes_df):,}")
    cuis_with_stypes = stypes_df['CUI'].nunique()
    print(f"   Unique CUIs with types: {cuis_with_stypes:,}")
    print(f"   Percent covered: {100*cuis_with_stypes/len(cuis_df):.1f}%")
    print(f"   Unique semantic types: {stypes_df['semantic_type'].nunique()}")
    
    # Term statistics
    print(f"\n4. TERMS:")
    print(f"   Total term entries: {len(terms_df):,}")
    cuis_with_terms = terms_df['CUI'].nunique()
    print(f"   Unique CUIs with terms: {cuis_with_terms:,}")
    print(f"   Percent covered: {100*cuis_with_terms/len(cuis_df):.1f}%")
    preferred = terms_df['is_preferred'].sum()
    print(f"   Preferred terms: {preferred:,}")
    print(f"   Synonym terms: {len(terms_df) - preferred:,}")
    print(f"   Average synonyms per CUI: {len(terms_df)/cuis_with_terms:.1f}")
    
    # Coverage summary
    print(f"\n5. OVERALL COVERAGE:")
    fully_covered = len(cuis_df[
        (cuis_df['CUI'].isin(defs_df['CUI'])) &
        (cuis_df['CUI'].isin(stypes_df['CUI'])) &
        (cuis_df['CUI'].isin(terms_df['CUI']))
    ])
    print(f"   CUIs with ALL data (defs + types + terms): {fully_covered:,}")
    print(f"   Percent: {100*fully_covered/len(cuis_df):.1f}%")


In [ ]:
def main():
    """
    Execute the UMLS preparation pipeline - save as CSVs for inspection.
    """
    print("\n" + "="*70)
    print("UMLS DATASET PREPARATION - TABULAR FORMAT (CSVs)")
    print("="*70)
    print(f"Project root: {PROJECT_ROOT}")
    print(f"UMLS META directory: {UMLS_META}")
    print(f"Output directory: {OUTPUT_DIR}")
    
    # Verify UMLS files exist
    required_files = ['MRCONSO.RRF', 'MRDEF.RRF', 'MRSTY.RRF']
    for filename in required_files:
        filepath = UMLS_META / filename
        if not filepath.exists():
            raise FileNotFoundError(f"Required file not found: {filepath}")
        print(f"✓ {filename} found ({filepath.stat().st_size / 1e9:.1f} GB)")
    
    try:
        # Step 1: Extract CUIs
        cuis_df, chest_cuis_set = extract_chest_cuis()
        
        # Step 2: Load definitions
        defs_df = load_definitions(chest_cuis_set)
        
        # Step 3: Load semantic types
        stypes_df = load_semantic_types(chest_cuis_set)
        
        # Step 4: Load terms
        terms_df = load_preferred_terms(chest_cuis_set)
        
        # Summary report
        print_summary_report(cuis_df, defs_df, stypes_df, terms_df)
        
        print("\n" + "="*70)
        print("✓ PIPELINE COMPLETED SUCCESSFULLY")
        print("="*70)
        print("\nOutput files in data/processed_data/:")
        print("  1. 01_chest_radiology_cuis.csv")
        print("  2. 02_definitions.csv")
        print("  3. 03_semantic_types.csv")
        print("  4. 04_terms.csv")
        
        
    except Exception as e:
        print(f"\n✗ ERROR: {e}")
        import traceback
        traceback.print_exc()
        raise

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
if __name__ == "__main__":
    main()

In [ ]:
def main():
    print("\n" + "="*70)
    print("CORRECTED UMLS EXTRACTION - WITH PROPER FILTERING")
    print("="*70)
    
    try:
        # Step 1: Extract with semantic type filtering
        cuis_df, chest_cuis_set = extract_and_filter_cuis()
        
        # Step 2: Load definitions
        defs_df = load_definitions(chest_cuis_set)
        
        # Step 3: Load terms
        terms_df = load_preferred_terms(chest_cuis_set)
        

        print(f"\nFinal output files:")
        print(f"  1. 01_chest_radiology_cuis_filtered.csv  ({len(cuis_df):,} CUIs)")
        print(f"  2. 02_definitions.csv                    ({len(defs_df):,} definitions)")
        print(f"  3. 03_terms.csv                          ({len(terms_df):,} terms)")
        
    except Exception as e:
        print(f"\n✗ ERROR: {e}")
        import traceback
        traceback.print_exc()
        raise


if __name__ == "__main__":
    main()

In [ ]:
def main():
    print("\n" + "="*70)
    print("CORRECTED UMLS EXTRACTION - FINAL VERSION")
    print("="*70)
    
    try:
        # Step 1: Extract with semantic type filtering
        cuis_df, chest_cuis_set = extract_and_filter_cuis()
        
        # Step 2: Load definitions
        defs_df = load_definitions(chest_cuis_set)
        
        # Step 3: Load terms
        terms_df = load_preferred_terms(chest_cuis_set)
        
        print("\n" + "="*70)
        print("✓ EXTRACTION COMPLETE")
        print("="*70)
        print(f"\nFinal output files:")
        print(f"  1. 01_chest_radiology_cuis_filtered.csv  ({len(cuis_df):,} CUIs)")
        print(f"  2. 02_definitions.csv                    ({len(defs_df):,} definitions)")
        print(f"  3. 03_terms.csv                          ({len(terms_df):,} terms)")
        
    except Exception as e:
        print(f"\n✗ ERROR: {e}")
        import traceback
        traceback.print_exc()
        raise


if __name__ == "__main__":
    main()

# DEBUG

### MRSTY

In [ ]:
ROOT = Path.cwd().parent
META = ROOT /"data"/"raw_data"/"umls"/"umls_2026AA_full"/"2026AA"/"META"

In [ ]:
def ram():
    return f"{psutil.Process().memory_info().rss/1e9:.2f} GB"

STY_COLS = ['CUI', 'TUI', 'STN', 'STY', 'ATUI', 'CVF']
mrsty = pd.read_csv(META/"MRSTY.RRF", sep="|", header=None,names=STY_COLS, index_col=False, dtype=str, encoding="utf-8")

In [ ]:
print(mrsty.shape, ram())
print(mrsty.head(3))
print("CUI format ok:", mrsty["CUI"].str.match(r"^C\d{7}$").mean())
print("TUI format ok:", mrsty["TUI"].str.match(r"^T\d{3}$").mean())
print(mrsty["TUI"].value_counts().head(10))

In [ ]:
ALLOWED_TUI = {
    "T017", "T023", "T029", "T030", "T024",        # anatomy
    "T033", "T184", "T034",                        # findings, signs, results
    "T047", "T046", "T191", "T037", "T190", "T020",# disorders
    "T060", "T061",                                # procedures
    "T074",                                        # devices (tubes, lines)
}

keep_cuis = set(mrsty.loc[mrsty["TUI"].isin(ALLOWED_TUI), "CUI"])
print("CUIs with an allowed type:", len(keep_cuis))
print("Types found in data but not allowed:",
      sorted(ALLOWED_TUI - set(mrsty["TUI"])))   # should be an empty list

### MRCONSO

In [ ]:
MRCONSO_COLS = ["CUI","LAT","TS","LUI","STT","SUI","ISPREF","AUI","SAUI",
                "SCUI","SDUI","SAB","TTY","CODE","STR","SRL","SUPPRESS","CVF"]
df = pd.read_csv(Path(r"D:\Project\Radiology_RAG\data\processed_data\filtered_data\MRCONSO_ENG_N.RRF"), sep="|", header=None, names=MRCONSO_COLS, usecols=["CUI", "SAB", "TTY", "STR"], index_col=False, quoting=3, 
                 dtype={"CUI": str, "STR": str, "SAB": "category", "TTY": "category"}, encoding="utf-8")

In [ ]:
print(df.shape, "| RAM", ram())
print("Unique CUIs:", df["CUI"].nunique())
print("Null STR   :", df["STR"].isna().sum())

print("\nTop 20 source vocabularies (SAB):")
print(df["SAB"].value_counts().head(20))

print("\nRows for the sources we care about:")
for sab in ["CHV", "MSH", "SNOMEDCT_US", "NCI", "MTH", "MEDLINEPLUS"]:
    sub = df[df["SAB"] == sab]
    print(f"  {sab:12s} rows={len(sub):>9,}  CUIs={sub['CUI'].nunique():>8,}")

print("\nTop 15 term types (TTY):")
print(df["TTY"].value_counts().head(15))

In [ ]:
df = df.dropna(subset=["STR"])
chv = df[df["SAB"] == "CHV"]

print("CHV term types:")
print(chv["TTY"].value_counts())

print("\nStrings per CHV CUI:")
print(chv.groupby("CUI").size().describe())

print("\nRandom CHV rows:")
print(chv.sample(15, random_state=42)[["CUI", "TTY", "STR"]].to_string())

# Do CHV concepts also have clinical names in the fallback sources?
chv_cuis = set(chv["CUI"])
for sab in ["MSH", "SNOMEDCT_US", "NCI"]:
    other = set(df.loc[df["SAB"] == sab, "CUI"])
    print(f"{sab:12s} overlap with CHV CUIs: {len(chv_cuis & other):,}")

# Spot-check chest terms
pat = r"\b(?:pneumothorax|pleural effusion|atelectasis|lung|pneumonia)\b"
chest_chv = chv[chv["STR"].str.lower().str.contains(pat, regex=True)]
print("\nChest-term CHV rows:", len(chest_chv))
print(chest_chv.sort_values("CUI")[["CUI", "TTY", "STR"]].head(30).to_string())

In [ ]:
SEEDS = [
    "lung", "pleura", "pleural space", "mediastinum", "heart", "thorax",
    "chest wall", "rib", "diaphragm", "trachea", "bronchus", "hilum",
    "pneumothorax", "pleural effusion", "atelectasis", "consolidation",
    "cardiomegaly", "pulmonary edema", "pulmonary nodule", "lung mass",
    "emphysema", "pulmonary fibrosis", "pneumonia", "pulmonary infiltrate",
    "rib fracture", "lung cancer", "pulmonary embolism",
    "chest x-ray", "chest ct", "endotracheal tube", "central venous catheter",
]
seed_set = set(SEEDS)

m = df[df["STR"].str.lower().isin(seed_set)].copy()
m = m[m["CUI"].isin(keep_cuis)]
m["seed"] = m["STR"].str.lower()

sty_names = (mrsty[mrsty["CUI"].isin(m["CUI"].unique())]
             .groupby("CUI")["STY"].agg("; ".join))
chv_pt  = chv[chv["TTY"] == "PT"].groupby("CUI")["STR"].agg(" | ".join)
chv_cnt = chv.groupby("CUI").size()

summary = (m.groupby(["seed", "CUI"])
           .agg(sources=("SAB", lambda s: ",".join(sorted(set(s)))))
           .reset_index())
summary["sty"]    = summary["CUI"].map(sty_names)
summary["chv_PT"] = summary["CUI"].map(chv_pt)
summary["n_chv"]  = summary["CUI"].map(chv_cnt).fillna(0).astype(int)

print(summary.to_string(max_colwidth=40))
print("\nSeeds with NO match:", sorted(seed_set - set(summary["seed"])))
print("Seeds with multiple CUIs:",
      summary.groupby("seed")["CUI"].nunique().loc[lambda s: s > 1].index.tolist())
print("Seeds with no CHV coverage:",
      sorted(set(summary["seed"]) - set(summary.loc[summary["n_chv"] > 0, "seed"])))

In [ ]:
# 1) Every CUI with the exact string "lung", before any keep_cuis filter
lung_rows = df[df["STR"].str.lower() == "lung"]
print(lung_rows.groupby("CUI").size())
for cui in lung_rows["CUI"].unique():
    print(cui, "| in keep_cuis:", cui in keep_cuis,
          "|", mrsty.loc[mrsty["CUI"] == cui, "STY"].tolist())

# 2) Is C0024109 in the file at all, under any string?
target = "C0024109"
print("\nRows for", target, "in df:", (df["CUI"] == target).sum())
print("In mrsty:", mrsty.loc[mrsty["CUI"] == target, ["TUI", "STY"]].values.tolist())
print(df.loc[df["CUI"] == target, ["SAB", "TTY", "STR"]].head(10))

In [ ]:
FILT = ROOT / "data" / "processed_data" / "filtered_data"
def scan_cui(path, target, label):
    found = []
    with open(path, encoding="utf-8", newline="") as f:
        for line in f:
            if line.startswith(target + "|"):
                found.append(line.split("|"))
            elif found:            # files are sorted by CUI, so stop after the block
                break
    print(f"\n{label}: {len(found)} lines for {target}")
    if found:
        from collections import Counter
        print(" LAT      :", Counter(p[1] for p in found))
        print(" SUPPRESS :", Counter(p[16] for p in found))
        print(" SAB      :", Counter(p[11] for p in found).most_common(6))
        print(" samples  :", [p[14] for p in found[:6]])

scan_cui(META / "MRCONSO.RRF", "C0024109", "RAW MRCONSO")
scan_cui(FILT / "MRCONSO_ENG.RRF", "C0024109", "Earlier filtered MRCONSO_ENG")

# How many CUIs with an allowed type are missing from the N-filtered file?
missing = keep_cuis - set(df["CUI"])
print("\nAllowed-type CUIs total  :", len(keep_cuis))
print("Missing from filtered df :", len(missing))
print(mrsty[mrsty["CUI"].isin(missing) & mrsty["TUI"].isin(ALLOWED_TUI)]
      ["STY"].value_counts().head(10))

In [ ]:
from collections import Counter

RAW = META / "MRCONSO.RRF"
NEW = FILT / "MRCONSO_ENG_N_v2.RRF"

raw_total = eng_total = kept = bad = lung_lines = 0
sab_kept = Counter()
cuis_v2 = set()

with open(RAW, encoding="utf-8", newline="") as fin, \
     open(NEW, "w", encoding="utf-8", newline="") as fout:
    for line in fin:
        raw_total += 1
        p = line.split("|", 17)          # need only the first 17 fields
        if len(p) < 17:
            bad += 1
            continue
        if p[1] != "ENG":
            continue
        eng_total += 1
        if p[16] == "N":
            fout.write(line)
            kept += 1
            sab_kept[p[11]] += 1
            cuis_v2.add(p[0])
            if p[0] == "C0024109":
                lung_lines += 1
        if raw_total % 3_000_000 == 0:
            print(f"{raw_total:,} lines | kept {kept:,} | RAM {ram()}")

print(f"\nRaw lines       : {raw_total:,}")
print(f"ENG lines       : {eng_total:,}   (earlier file had 10,754,012)")
print(f"ENG + SUPPRESS=N: {kept:,}   (earlier N file had 9,110,885)")
print(f"Malformed       : {bad:,}")
print(f"Lung C0024109 lines kept: {lung_lines}")
print(f"Output size (GB): {NEW.stat().st_size / 1e9:.2f}")

# What did the earlier file lose?
old_cuis = set(df["CUI"])
lost = cuis_v2 - old_cuis
print(f"\nCUIs in new file but absent from old: {len(lost):,}")
print(f"  ...of which allowed-type: {len(lost & keep_cuis):,}")
print(f"Allowed-type CUIs still missing (no English, non-suppressed name): "
      f"{len(keep_cuis - cuis_v2):,}")

old_sab = df["SAB"].value_counts()
diff = {s: n - int(old_sab.get(s, 0)) for s, n in sab_kept.items()}
print("\nSources with the most rows lost by the earlier filter:")
for s, d in sorted(diff.items(), key=lambda x: -x[1])[:15]:
    print(f"  {s:14s} {d:>10,}")

In [ ]:
del df, chv
gc.collect()

df = pd.read_csv(
    NEW, sep="|", header=None, names=MRCONSO_COLS,
    usecols=["CUI", "SAB", "TTY", "STR"], index_col=False, quoting=3,
    dtype={"CUI": str, "STR": str, "SAB": "category", "TTY": "category"},
    encoding="utf-8",
)
df = df.dropna(subset=["STR"])
chv = df[df["SAB"] == "CHV"]
print(df.shape, "| RAM", ram(), "| CHV CUIs:", chv["CUI"].nunique())

# Lung sanity check
print("\nLung (C0024109) rows by source:")
print(df[df["CUI"] == "C0024109"].groupby("SAB", observed=True).size()
      .sort_values(ascending=False).head(8))
print(chv[chv["CUI"] == "C0024109"][["TTY", "STR"]])

# The 27 allowed-type concepts the old file had lost
lost_allowed = sorted(lost & keep_cuis)
names = df[df["CUI"].isin(lost_allowed)].groupby("CUI")["STR"].first()
sty1 = mrsty[mrsty["CUI"].isin(lost_allowed)].groupby("CUI")["STY"].first()
print("\nRecovered allowed-type concepts:")
print(pd.DataFrame({"name": names, "sty": sty1}).to_string())

### Miscellaneous

In [ ]:
mrsty_file = UMLS_META / "MRSTY.RRF"

all_stypes = {}
sample_cuis = set()

for chunk_num, chunk in enumerate(
    pd.read_csv(
        mrsty_file,
        sep='|',
        header=None,
        usecols=[0, 3],
        names=['CUI', 'STY'],
        chunksize=100000,
        dtype={'CUI': str, 'STY': str},
        engine='c'
    ),
    1
):
    for _, row in chunk.iterrows():
        cui = row['CUI'].strip()  # ← TRY STRIPPING WHITESPACE
        sty = row['STY'].strip()  # ← TRY STRIPPING WHITESPACE
        
        if cui not in all_stypes:
            all_stypes[cui] = []
        all_stypes[cui].append(sty)
        
        if len(sample_cuis) < 20:
            sample_cuis.add(cui)
    
    if chunk_num == 1:  # Just load first chunk for speed
        break

print(f"\nLoaded {len(all_stypes):,} CUIs with semantic types")


print("\n2. SAMPLE DATA FROM all_stypes:")
print("-" * 70)

for i, cui in enumerate(sorted(list(sample_cuis))[:10]):
    stypes = all_stypes[cui]
    print(f"  CUI: {repr(cui):<20} | STY: {stypes}")


CHEST_RELEVANT_STYPES = {'T003', 'T004', 'T005', 'T007'}

print(f"\n3. CHECKING FOR CHEST TYPES: {CHEST_RELEVANT_STYPES}")
print("-" * 70)

chest_count = 0
for cui, stypes in all_stypes.items():
    has_chest = any(st in CHEST_RELEVANT_STYPES for st in stypes)
    if has_chest:
        chest_count += 1
        if chest_count <= 10:
            print(f"  {cui}: {stypes}")

print(f"\n  Total CUIs with chest types (T003/T004/T005/T007): {chest_count:,}")


print("\n4. TESTING MRCONSO MATCHING...")
print("-" * 70)

MRCONSO_FILTERED = PROJECT_ROOT / "data/processed_data/filtered_data/MRCONSO.RRF"

# Load first few rows
sample_mrconso = pd.read_csv(
    MRCONSO_FILTERED,
    sep='|',
    header=None,
    usecols=[0, 14],
    names=['CUI', 'STR'],
    nrows=1000,
    dtype={'CUI': str, 'STR': str},
    engine='c'
)

print(f"Loaded {len(sample_mrconso):,} sample terms from MRCONSO")

# Check if CUIs in MRCONSO exist in all_stypes
cuis_in_mrconso = sample_mrconso['CUI'].unique()
matched = 0
unmatched = 0

for cui in cuis_in_mrconso[:20]:
    if cui in all_stypes:
        matched += 1
        term = sample_mrconso[sample_mrconso['CUI'] == cui]['STR'].iloc[0]
        stypes = all_stypes[cui]
        has_chest = any(st in CHEST_RELEVANT_STYPES for st in stypes)
        print(f"  ✓ {cui}: {term[:40]:<40} | Types: {stypes} | Has chest: {has_chest}")
    else:
        unmatched += 1
        term = sample_mrconso[sample_mrconso['CUI'] == cui]['STR'].iloc[0]
        print(f"  ✗ {cui}: {term[:40]:<40} | NOT IN all_stypes")

print(f"\nMatched: {matched} / {len(cuis_in_mrconso[:20])}")
print(f"Unmatched: {unmatched} / {len(cuis_in_mrconso[:20])}")

# ============================================================================
# STEP 5: Check for whitespace issues
# ============================================================================

print("\n5. CHECKING FOR WHITESPACE/FORMATTING ISSUES...")
print("-" * 70)

for cui in list(all_stypes.keys())[:5]:
    stypes = all_stypes[cui]
    for sty in stypes:
        print(f"  CUI {cui} | STY repr: {repr(sty):<20} | In set: {sty in CHEST_RELEVANT_STYPES}")

In [ ]:
"""
DEBUG: Keyword matching - what terms are being matched?
"""

import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("D:/Project/Radiology_RAG")
MRCONSO_FILTERED = PROJECT_ROOT / "data/processed_data/filtered_data/MRCONSO.RRF"

CHEST_KEYWORDS = [
    'lung', 'lungs', 'pulmonary', 'pneumonia', 'pneumothorax',
    'bronch', 'trachea', 'mediastin', 'pleura', 'pleural',
    'diaphragm', 'rib', 'ribs', 'heart', 'cardiac', 'cardiomegaly',
    'thorax', 'thoracic', 'chest',
    'consolidation', 'infiltrate', 'atelectasis', 'emphysema',
    'pulmonary edema', 'pleural effusion', 'fibrosis',
    'tuberculosis', 'asthma', 'copd', 'bronchitis',
    'aspiration', 'sarcoidosis', 'cystic fibrosis',
    'opacity', 'opacities', 'hyperinflation', 'cavitary',
    'nodule', 'mass', 'pleural thickening'
]

print("="*80)
print("DEBUGGING KEYWORD MATCHING")
print("="*80)

# Load first 100K rows to sample
print("\nLoading first 100K terms from MRCONSO...")
mrconso = pd.read_csv(
    MRCONSO_FILTERED,
    sep='|',
    header=None,
    usecols=[0, 14],
    names=['CUI', 'STR'],
    nrows=100000,
    dtype={'CUI': str, 'STR': str},
    engine='c'
)

mrconso['STR_lower'] = mrconso['STR'].str.lower()

# Find which keyword matched each term
matches_by_kw = {kw: [] for kw in CHEST_KEYWORDS}

print(f"Scanning {len(mrconso):,} terms for matches...")

for _, row in mrconso.iterrows():
    term_lower = row['STR_lower']
    for kw in CHEST_KEYWORDS:
        if kw in term_lower:
            matches_by_kw[kw].append({
                'CUI': row['CUI'],
                'term': row['STR'],
                'term_lower': term_lower
            })
            break  # Only record first match

# ============================================================================
# ANALYSIS 1: Distribution
# ============================================================================

print("\n" + "="*80)
print("KEYWORD MATCH DISTRIBUTION (top 15)")
print("="*80)

sorted_matches = sorted(matches_by_kw.items(), key=lambda x: len(x[1]), reverse=True)

for kw, matches in sorted_matches[:15]:
    print(f"  {kw:25s}: {len(matches):6,} matches")

# ============================================================================
# ANALYSIS 2: Spot-check high-volume keywords
# ============================================================================

print("\n" + "="*80)
print("SPOT-CHECK: SAMPLE MATCHES")
print("="*80)

# Check potentially problematic keywords
problem_keywords = ['heart', 'mass', 'nodule', 'consolidation', 'bronch']

for kw in problem_keywords:
    matches = matches_by_kw[kw]
    if matches:
        print(f"\nKeyword: '{kw}' ({len(matches):,} matches)")
        print("-" * 80)
        print("  Sample terms:")
        for match in matches[:15]:
            print(f"    {match['CUI']:12s} | {match['term'][:60]}")
        
        # Look for suspicious ones
        suspicious = [m for m in matches if any(
            bad in m['term_lower'] for bad in 
            ['sweethe', 'lionhe', 'heartthrob', 'lionheart', 'heartburn', 'chemistry', 'spectrum']
        )]
        
        if suspicious:
            print(f"\n  ⚠️  Suspicious matches ({len(suspicious)}):")
            for m in suspicious[:5]:
                print(f"    - {m['term']}")

# ============================================================================
# ANALYSIS 3: Check for substring collisions
# ============================================================================

print("\n" + "="*80)
print("POTENTIAL SUBSTRING COLLISION ANALYSIS")
print("="*80)

print("\nKeywords that might be substrings of other keywords:")
for i, kw1 in enumerate(CHEST_KEYWORDS):
    for kw2 in CHEST_KEYWORDS[i+1:]:
        if kw1 in kw2 or kw2 in kw1:
            print(f"  '{kw1}' ⊂ '{kw2}'")

# ============================================================================
# ANALYSIS 4: Check word boundary issues
# ============================================================================

print("\n" + "="*80)
print("WORD BOUNDARY ISSUES")
print("="*80)

test_cases = [
    "heart of palm",
    "sweetheart syndrome", 
    "mass spectrometry",
    "massive hemorrhage",
    "nodelike structure",
    "bronchial tree",
    "bronchoconstriction"
]

print("\nTesting simple substring matching:")
for term in test_cases:
    term_lower = term.lower()
    matched = [kw for kw in CHEST_KEYWORDS if kw in term_lower]
    status = "MATCH" if matched else "NO MATCH"
    print(f"  {status:8s} | {term:35s} | Matched: {matched}")

print("\n" + "="*80)

In [ ]:
"""
DEBUG VERSION - Show what's happening at each stage
"""

import pandas as pd
import numpy as np
from pathlib import Path
import gc


CHUNK_SIZE = 100000

CHEST_RELEVANT_STYPES = {'T003', 'T004', 'T005', 'T007'}

CHEST_KEYWORDS_FIXED = [
    r'\blung\b', r'\blungs\b', r'\bpulmonary\b',
    r'\bthorax\b', r'\bthoracic\b', r'\bchest\b',
    r'\bmediastin', r'\bpleura', r'\bpleural\b',
    r'\bdiaphragm\b', r'\bribs?\b', r'\btrachea\b',
    r'\bpneumonia\b', r'\bpneumothorax\b',
    r'\bconsolidation\b', r'\binfiltrat',
    r'\batelectasis\b', r'\bemphysema\b',
    r'\bfibrosis\b', r'\bcavit',
    r'\bpleural effusion\b', r'\bpulmonary edema\b',
    r'\btuberculosis\b', r'\btubercul',
    r'\bopacit', r'\bhyperinflation\b',
    r'\bpulmonary nodule\b', r'\bpulmonary mass\b',
    r'\bsarcoidosis\b', r'\bcystic fibrosis\b',
    r'\bcardiomegaly\b', r'\bcardiac silhouette\b',
    r'\bhilum\b', r'\bhila\b', r'\bhilar\b',
]

pattern = '|'.join(CHEST_KEYWORDS_FIXED)

print("\n" + "="*70)
print("DEBUG VERSION - UMLS EXTRACTION")
print("="*70)

# STEP 1a: Load semantic types
print("\n1a. Loading semantic types...")
mrsty_file = UMLS_META / "MRSTY.RRF"

all_stypes = {}
for chunk_num, chunk in enumerate(
    pd.read_csv(
        mrsty_file,
        sep='|',
        header=None,
        usecols=[0, 2],
        names=['CUI', 'STY'],
        chunksize=CHUNK_SIZE,
        dtype={'CUI': str, 'STY': str},
        engine='c'
    ),
    1
):
    for _, row in chunk.iterrows():
        cui = row['CUI'].strip()
        sty = row['STY'].strip()
        if cui not in all_stypes:
            all_stypes[cui] = []
        all_stypes[cui].append(sty)

print(f"   Loaded {len(all_stypes):,} CUIs")

# Check samples
sample_cuis = list(all_stypes.keys())[:5]
print(f"\n   Sample CUIs with types:")
for cui in sample_cuis:
    print(f"     {cui}: {all_stypes[cui]}")

# STEP 1b: Test keyword matching on first chunk only
print("\n1b. Testing keyword matching on FIRST CHUNK ONLY...")

mrconso_file = MRCONSO_FILTERED

chunk = pd.read_csv(
    mrconso_file,
    sep='|',
    header=None,
    usecols=[0, 14],
    names=['CUI', 'STR'],
    nrows=50000,  # Just first 50K
    dtype={'CUI': str, 'STR': str},
    engine='c'
)

print(f"   Loaded {len(chunk):,} terms")

chunk['STR_lower'] = chunk['STR'].str.lower()

print(f"\n   Testing regex pattern...")
print(f"   Pattern: {pattern[:100]}...")

try:
    keyword_mask = chunk['STR_lower'].str.contains(
        pattern, 
        na=False, 
        regex=True,
        case=False
    )
    print(f"   ✓ Regex works!")
    print(f"   Matches found: {keyword_mask.sum():,}")
except Exception as e:
    print(f"   ✗ Regex error: {e}")
    keyword_mask = None

# STEP 1c: Check if matched CUIs are in all_stypes
if keyword_mask is not None and keyword_mask.sum() > 0:
    print(f"\n1c. Checking if matched CUIs have semantic types...")
    
    matching = chunk[keyword_mask].copy()
    
    in_stypes = 0
    not_in_stypes = 0
    has_chest_type = 0
    
    for _, row in matching.iterrows():
        cui = row['CUI']
        
        if cui in all_stypes:
            in_stypes += 1
            stypes = all_stypes[cui]
            
            # Check if has chest type
            if any(st in CHEST_RELEVANT_STYPES for st in stypes):
                has_chest_type += 1
        else:
            not_in_stypes += 1
    
    print(f"   Matched CUIs in all_stypes: {in_stypes:,}")
    print(f"   Matched CUIs NOT in all_stypes: {not_in_stypes:,}")
    print(f"   With chest-relevant types (T003/T004/T005/T007): {has_chest_type:,}")
    
    # Show sample matches
    print(f"\n   Sample MATCHING terms:")
    for _, row in matching.head(10).iterrows():
        cui = row['CUI']
        term = row['STR']
        if cui in all_stypes:
            stypes = all_stypes[cui]
            has_chest = any(st in CHEST_RELEVANT_STYPES for st in stypes)
            print(f"     {cui} | {term[:50]:50s} | Types: {stypes} | Chest: {has_chest}")
        else:
            print(f"     {cui} | {term[:50]:50s} | NOT IN STYPES")

else:
    print(f"\n✗ No keyword matches found or regex failed!")

print("\n" + "="*70)